# Ball Balance — Train + Eval
Set `TASK` in cell 1, hit Run All, walk away.
Trains the policy, evaluates it, records a GIF, and saves everything to the log dir.

In [ ]:
# ── 1. Config ─────────────────────────────────────────────────────────────
import os

ON_KAGGLE = os.path.exists('/kaggle')

BRANCH         = 'final-eval'
TASK           = 'BallBalance-SingleArm-v0'   # ← change per run
# Options:
#   BallBalance-SingleArm-v0
#   BallBalance-DualArm-v0
#   BallBalance-DualArm-Attention-v0

N_ENVS         = 2048 if ON_KAGGLE else 4
MAX_ITERATIONS = 1000 if ON_KAGGLE else 5
ACTION_DELTA   = 0.3

# Eval settings
N_EVAL_ENVS  = 8 if ON_KAGGLE else 2
NUM_EPISODES = 50 if ON_KAGGLE else 5    # completed episodes for eval

# GIF settings
GIF_STEPS      = 500
GIF_DOWNSAMPLE = 3    # keep every Nth frame
GIF_FPS        = 20

REPO_DIR = '/kaggle/working/bimanual-project' if ON_KAGGLE else os.path.abspath('.')

print(f'ON_KAGGLE: {ON_KAGGLE}')
print(f'TASK:      {TASK}')
print(f'REPO_DIR:  {REPO_DIR}')

In [ ]:
# ── 2. Clone / pull repo  (Kaggle only) ───────────────────────────────────
if ON_KAGGLE:
    from kaggle_secrets import UserSecretsClient
    import subprocess

    token = UserSecretsClient().get_secret('GITHUB_TOKEN')

    if not os.path.exists(REPO_DIR):
        result = subprocess.run([
            'git', 'clone', '--branch', BRANCH,
            f'https://{token}@github.com/sharana-sabesan09/bimanual-project.git',
            REPO_DIR,
        ], capture_output=True, text=True)
        print(result.stdout or result.stderr)
    else:
        result = subprocess.run(
            ['git', '-C', REPO_DIR, 'pull'],
            capture_output=True, text=True
        )
        print(result.stdout)

    commit = subprocess.run(
        ['git', '-C', REPO_DIR, 'log', '-1', '--pretty=%h %s'],
        capture_output=True, text=True
    )
    print('Commit:', commit.stdout.strip())
else:
    import subprocess
    commit = subprocess.run(
        ['git', 'log', '-1', '--pretty=%h %s'],
        capture_output=True, text=True
    )
    print('Local repo — commit:', commit.stdout.strip())

In [ ]:
# ── 3. Install dependencies  (Kaggle only) ────────────────────────────────
if ON_KAGGLE:
    result = subprocess.run(
        ['pip', 'install', '-r', os.path.join(REPO_DIR, 'requirements.txt')],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print('pip STDERR:')
        print(result.stderr[-3000:])
    else:
        print('Dependencies installed.')
else:
    print('Local — skipping installs.')

In [ ]:
# ── 4. Setup paths ────────────────────────────────────────────────────────
import sys

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

print(f'Working dir: {os.getcwd()}')

In [ ]:
# ── 5. Verify GPU ─────────────────────────────────────────────────────────
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU:  {props.name}')
    print(f'VRAM: {props.total_memory / 1e9:.1f} GB')

In [ ]:
# ── 6. Train ──────────────────────────────────────────────────────────────
import glob

cmd = (
    f"python scripts/rsl_rl/train.py"
    f" --task {TASK}"
    f" -e {TASK}"
    f" -n {N_ENVS}"
    f" --max_iterations {MAX_ITERATIONS}"
    f" --action_delta {ACTION_DELTA}"
    f" --headless"
)
print(f'Running: {cmd}')
!{cmd}

# Grab the log dir that train.py just created
runs = sorted(glob.glob(os.path.join(REPO_DIR, 'logs', TASK, '*')))
LOG_DIR  = runs[-1]
CKPT     = os.path.join(LOG_DIR, f'model_{MAX_ITERATIONS}.pt')
print(f'Log dir:    {LOG_DIR}')
print(f'Checkpoint: {CKPT}')

In [ ]:
# ── 7. Training curves ────────────────────────────────────────────────────
import matplotlib.pyplot as plt
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

ea = EventAccumulator(LOG_DIR)
ea.Reload()

def tb_extract(tag):
    events = ea.Scalars(tag)
    return [e.step for e in events], [e.value for e in events]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

steps, vals = tb_extract('Train/mean_reward')
axes[0].plot(steps, vals)
axes[0].set_title('Mean Reward')
axes[0].set_xlabel('Iteration')
axes[0].grid(True)

steps, vals = tb_extract('Train/mean_episode_length')
axes[1].plot(steps, vals, color='orange')
axes[1].axhline(500, color='gray', linestyle='--', label='max ep length')
axes[1].set_title('Mean Episode Length')
axes[1].set_xlabel('Iteration')
axes[1].legend()
axes[1].grid(True)

plt.suptitle(TASK, fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(LOG_DIR, 'training_curves.png'), dpi=150)
plt.show()

In [ ]:
# ── 8. Evaluate ───────────────────────────────────────────────────────────
import importlib
import pickle
import statistics
from pathlib import Path
import numpy as np
import genesis as gs
import gymnasium as gym
import source  # triggers gym.register() calls

from rsl_rl.runners import OnPolicyRunner
from scripts.rsl_rl.vec_env import RslRlVecEnvWrapper
from scripts.rsl_rl.eval import per_goal_metrics, GOAL_SWITCH_PERIOD

def _resolve(entry_point):
    module_path, attr = entry_point.rsplit(':', 1)
    return getattr(importlib.import_module(module_path), attr)

gs.init(backend=gs.gpu, precision='32', logging_level='warning')

ckpt = Path(CKPT)
with open(ckpt.parent / 'train_cfg.pkl', 'rb') as f:
    train_cfg = pickle.load(f)

env_spec = gym.spec(TASK)
EnvClass = _resolve(env_spec.entry_point)
raw_env  = EnvClass(n_envs=N_EVAL_ENVS, show_viewer=False,
                    action_delta=ACTION_DELTA, debug=True)
env      = RslRlVecEnvWrapper(raw_env)

runner = OnPolicyRunner(env, train_cfg, str(ckpt.parent), device=gs.device)
runner.load(ckpt, map_location=torch.device('cpu'))
policy = runner.get_inference_policy(device=gs.device)

obs = env.reset()
completed, all_ep_dists = 0, []

with torch.no_grad():
    while completed < NUM_EPISODES:
        actions = policy(obs)
        obs, _, dones, _ = env.step(actions)
        for env_idx in dones.nonzero(as_tuple=True)[0]:
            all_ep_dists.append(raw_env._return_and_reset_debug(env_idx.item()))
            completed += 1
            if completed >= NUM_EPISODES:
                break

# Sim runs at 50 Hz (dt=0.02); 100 steps = 2 seconds
avg_dist   = statistics.fmean([statistics.fmean(d) for d in all_ep_dists])
after_2s   = [statistics.fmean(d[100:]) for d in all_ep_dists if len(d) > 100]
avg_ep_len = statistics.fmean([len(d) for d in all_ep_dists])

# Per-goal-switch
all_settle, all_success = [], []
for ep_dists in all_ep_dists:
    st, sf, _ = per_goal_metrics(ep_dists)
    all_settle.extend(st)
    all_success.extend(sf)

n_seg  = len(all_success)
n_succ = sum(all_success)
sr     = n_succ / n_seg if n_seg > 0 else float('nan')
penalty_settle = all_settle + [2.0] * (n_seg - n_succ)  # DNF penalty = full 2s window

EVAL_RESULTS = {
    'task':               TASK,
    'episodes':           completed,
    'success_rate_%':     sr * 100,
    'median_settle_s':    statistics.median(all_settle) if all_settle else float('nan'),
    'mean_settle_pen_s':  statistics.fmean(penalty_settle) if penalty_settle else float('nan'),
    'avg_dist_m':         avg_dist,
    'avg_dist_after2s_m': statistics.fmean(after_2s) if after_2s else float('nan'),
    'avg_ep_len':         avg_ep_len,
    'n_goal_segments':    n_seg,
    '_settle_times':      all_settle,
}

print(f'\nEVAL RESULTS — {TASK}')
print(f'  Success rate          : {sr*100:.1f}%  ({n_succ}/{n_seg} goal segments)')
print(f'  Median settle time    : {EVAL_RESULTS["median_settle_s"]:.3f} s')
print(f'  Mean settle (DNF=2s)  : {EVAL_RESULTS["mean_settle_pen_s"]:.3f} s')
print(f'  Avg dist from goal    : {avg_dist:.4f} m')
print(f'  Avg dist after 2s     : {EVAL_RESULTS["avg_dist_after2s_m"]:.4f} m')
print(f'  Avg episode length    : {avg_ep_len:.1f} steps')

In [ ]:
# ── 9. Eval plots ─────────────────────────────────────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Settle time histogram
ax = axes[0]
if all_settle:
    ax.hist(all_settle, bins=20, edgecolor='black')
    ax.axvline(statistics.median(all_settle), color='red', linestyle='--',
               label=f'median {statistics.median(all_settle):.2f}s')
    ax.legend()
ax.set_xlabel('Settle time (s)')
ax.set_ylabel('Count')
ax.set_title(f'Settle Time Distribution\n(success rate {sr*100:.1f}%)')
ax.grid(True, axis='y')

# Mean distance over time (average across episodes, aligned to episode start)
ax = axes[1]
max_len = max(len(d) for d in all_ep_dists)
padded  = np.array([d + [np.nan]*(max_len - len(d)) for d in all_ep_dists])
mean_dist = np.nanmean(padded, axis=0)
ax.plot(mean_dist)
for switch in range(0, max_len, GOAL_SWITCH_PERIOD):
    ax.axvline(switch, color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel('Timestep (50 Hz; goal switches every 2s / 100 steps)')
ax.set_ylabel('Mean XY dist to goal (m)')
ax.set_title('Avg Distance Over Episode\n(gray lines = goal switches)')
ax.grid(True)

plt.suptitle(TASK, fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(LOG_DIR, 'eval_plots.png'), dpi=150)
plt.show()

In [ ]:
# ── 10. Record GIF ────────────────────────────────────────────────────────
# Runs as a subprocess so Genesis can reinitialise with the camera attached.
import subprocess

GIF_PATH = os.path.join(LOG_DIR, f'{TASK}.gif')
record_script = os.path.join(REPO_DIR, 'scripts', 'rsl_rl', 'record_gif.py')

cmd = [
    'python', record_script,
    '--task',         TASK,
    '--checkpoint',   CKPT,
    '--out',          GIF_PATH,
    '--steps',        str(GIF_STEPS),
    '--downsample',   str(GIF_DOWNSAMPLE),
    '--fps',          str(GIF_FPS),
    '--action_delta', str(ACTION_DELTA),
]

print('Recording GIF...')
result = subprocess.run(cmd, capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout.strip())
else:
    print('ERROR:', result.stderr[-800:])

In [ ]:
# ── 11. Display GIF + save results ────────────────────────────────────────
import base64
from IPython.display import HTML, display

if os.path.exists(GIF_PATH):
    with open(GIF_PATH, 'rb') as f:
        data = base64.b64encode(f.read()).decode('ascii')
    display(HTML(
        f'<h3>{TASK}</h3>'
        f'<img src="data:image/gif;base64,{data}" style="height:350px"/>'
    ))
else:
    print('GIF not found — check cell 10 for errors.')

# Save eval numbers alongside the checkpoint
import json
results_path = os.path.join(LOG_DIR, 'eval_results.json')
save = {k: v for k, v in EVAL_RESULTS.items() if not k.startswith('_')}
with open(results_path, 'w') as f:
    json.dump(save, f, indent=2)
print(f'Results saved to: {results_path}')

# List all outputs
print(f'\nFiles in {LOG_DIR}:')
for fname in sorted(os.listdir(LOG_DIR)):
    size_mb = os.path.getsize(os.path.join(LOG_DIR, fname)) / 1e6
    print(f'  {fname}  ({size_mb:.1f} MB)')